In [3]:

#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
WVS Factor Analysis (country-level)
-----------------------------------
- Loads data-wvs.csv and map-wvs.json
- Cleans/standardizes items (Q[0-9]+), mean-imputes, and z-scores
- Parallel Analysis (PA) -> suggested k (capped at n-1)
- FactorAnalysis (sklearn) with varimax rotation
- Saves:
  * scree_parallel.png
  * factor_loadings_rotated.csv  (capacity-limited model, k = min(PA, n-1))
  * top_loadings.txt             (capacity-limited, top-10 per factor)
  * factor_scores.csv            (capacity-limited)
  * factor_loadings_k10.csv      (k=10 model)
  * top_loadings_k10.txt         (k=10 model)
  * factor_scores_k10.csv        (k=10 model)

Notes:
- With many items (p) vs. few countries (n), PA may yield large k. We cap at n-1.
- Pearson correlations used (approximation for ordinal items). For publication,
  consider polychoric FA using appropriate packages.
"""

import json
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import numpy.linalg as LA
import pandas as pd
from numpy.linalg import eigvalsh, pinv, slogdet
from sklearn.decomposition import FactorAnalysis


# ------------------------------- Utilities -------------------------------- #

def varimax(Phi: np.ndarray, gamma: float = 1.0, q: int = 50, tol: float = 1e-6):
    """
    Orthogonal varimax rotation (Kaiser 1958).
    Parameters
    ----------
    Phi : (p x k) array of loadings
    gamma : float, default 1.0 (varimax). Set to 0.0 for quartimax.
    q : int, max number of iterations
    tol : float, convergence criterion

    Returns
    -------
    Phi_rot : rotated loadings (p x k)
    R : rotation matrix (k x k)
    """
    p, k = Phi.shape
    R = np.eye(k)
    d = 0.0
    for _ in range(q):
        d_old = d
        Lambda = Phi @ R
        # Varimax update
        B = Phi.T @ (Lambda**3 - (gamma / p) * Lambda @ np.diag(np.diag(Lambda.T @ Lambda)))
        U, S, Vt = LA.svd(B)
        R = U @ Vt
        d = S.sum()
        if d_old != 0 and d / d_old < 1.0 + tol:
            break
    return Phi @ R, R


def kmo_bartlett(R: np.ndarray, n: int, ridge: float = 1e-6):
    """
    KMO and Bartlett's test (approx.) with a tiny ridge for numerical stability.
    Returns
    -------
    kmo : float
    chi2 : float (approximate)
    df_bartlett : int
    """
    # Regularize a touch to invert safely
    Rr = R + ridge * np.eye(R.shape[0])
    invR = pinv(Rr)

    # Partial correlations (off-diagonal)
    partial = -invR / np.sqrt(np.outer(np.diag(invR), np.diag(invR)))
    np.fill_diagonal(partial, 0.0)

    # Zero the diag for raw correlations, too
    Rc = R.copy()
    np.fill_diagonal(Rc, 0.0)

    num = (Rc**2).sum()
    den = num + (partial**2).sum()
    kmo = float(num / den) if den > 0 else float("nan")

    sign, logdet = slogdet(Rr)
    if sign <= 0:
        chi2 = float("nan")
    else:
        p = R.shape[0]
        chi2 = -(n - 1 - (2 * p + 5) / 6) * logdet
    df_bartlett = int(R.shape[0] * (R.shape[0] - 1) / 2)

    return kmo, chi2, df_bartlett


def parallel_analysis(X_std: pd.DataFrame, n_rep: int = 100, seed: int = 42, ridge: float = 1e-6):
    """
    Parallel Analysis against 95th percentile of random eigenvalues.
    Returns
    -------
    obs_eigs : array of observed eigenvalues (sorted desc)
    rand95 : array of 95th percentile eigenvalues (sorted desc)
    k_pa : int, count of observed > rand95
    k_kaiser : int, count of observed > 1.0
    """
    X = X_std.to_numpy()
    # Observed correlation
    R = np.corrcoef(X, rowvar=False) + ridge * np.eye(X.shape[1])
    obs_eigs = np.sort(eigvalsh(R))[::-1]

    rng = np.random.default_rng(seed)
    rand_eigs = np.zeros((n_rep, X.shape[1]))
    n, p = X.shape
    for i in range(n_rep):
        Z = rng.standard_normal(size=(n, p))
        Z = (Z - Z.mean(0)) / Z.std(0)
        Rz = np.corrcoef(Z, rowvar=False) + ridge * np.eye(p)
        rand_eigs[i, :] = np.sort(eigvalsh(Rz))[::-1]

    rand95 = np.percentile(rand_eigs, 95, axis=0)
    k_pa = int((obs_eigs > rand95).sum())
    k_kaiser = int((obs_eigs > 1.0).sum())
    return obs_eigs, rand95, k_pa, k_kaiser


def scree_plot(obs_eigs: np.ndarray, rand95: np.ndarray, out_path: Path):
    plt.figure(figsize=(8, 5))
    plt.plot(range(1, len(obs_eigs) + 1), obs_eigs, marker='o', label='Observed eigenvalues')
    plt.plot(range(1, len(rand95) + 1), rand95, '--', color='red', label='PA 95th percentile')
    plt.axhline(1.0, color='gray', linestyle=':', label='Kaiser = 1')
    plt.xlabel('Component number')
    plt.ylabel('Eigenvalue')
    plt.title('Scree & Parallel Analysis')
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


def fit_fa_with_rotation(X_std: pd.DataFrame, k: int, qmap: dict,
                         id_col: str | None,
                         model_tag: str,
                         out_dir: Path):
    """
    Fit FactorAnalysis(k) + varimax, save loadings, top-items, and scores.

    Parameters
    ----------
    X_std : standardized dataframe (n x p)
    k : number of factors
    qmap : dict mapping "Qxx" -> question text
    id_col : optional country id column found in raw data (to merge scores)
    model_tag : label for file names, e.g. "k10" or "rotated"
    out_dir : output directory
    """
    n, p = X_std.shape
    if k < 1:
        raise ValueError("k must be >= 1")
    if k >= n:
        warnings.warn(f"k={k} >= n={n}; reducing to k={n-1}")
        k = n - 1

    fa = FactorAnalysis(n_components=k, random_state=42)
    fa.fit(X_std)

    loadings = fa.components_.T  # p x k
    L_rot, R = varimax(loadings)

    # Rotated factor scores
    scores = fa.transform(X_std) @ R  # (n x k)

    # Build labeled loadings table
    codes = list(X_std.columns)
    labels = [qmap.get(c, c) for c in codes]
    load_df = pd.DataFrame(
        L_rot,
        index=pd.MultiIndex.from_arrays([codes, labels], names=['Code', 'Question']),
        columns=[f'Factor{j+1}' for j in range(k)]
    )
    load_df['Communality'] = (L_rot ** 2).sum(1)

    # Save full loadings
    load_path = out_dir / f'factor_loadings_{model_tag}.csv'
    load_df.to_csv(load_path, encoding='utf-8')

    # Save top 10 items per factor
    top_txt = out_dir / f'top_loadings_{model_tag}.txt'
    with open(top_txt, 'w', encoding='utf-8') as f:
        for j in range(k):
            col = f'Factor{j+1}'
            idx = load_df[col].abs().sort_values(ascending=False).head(10).index
            f.write(f"\n=== {col}: Top 10 items ===\n")
            f.write(load_df.loc[idx, [col]].to_string())
            f.write("\n")

    # Save factor scores (merge id if present)
    scores_df = pd.DataFrame(scores, columns=[f'Factor{j+1}' for j in range(k)])
    if id_col is not None and id_col in RAW.columns:
        scores_df.insert(0, id_col, RAW.loc[X_std.index, id_col].values)
    scores_path = out_dir / f'factor_scores_{model_tag}.csv'
    scores_df.to_csv(scores_path, index=False, encoding='utf-8')

    return load_path, top_txt, scores_path






In [5]:
CSV_PATH = Path("/data/demo_data/wvs/data-wvs.csv")
MAP_PATH = Path("/data/demo_data/wvs/map-wvs.json")
# ------------------------------ Main routine ------------------------------- #

if __name__ == "__main__":
    DATA = CSV_PATH
    MAP = MAP_PATH
    OUT = Path(".")
    OUT.mkdir(parents=True, exist_ok=True)

    # ---- Load data & codebook ----
    RAW = pd.read_csv(DATA)

    with open(MAP, "r", encoding="utf-8") as f:
        QMAP = json.load(f)


    # Item columns
    pattern = re.compile(r"^Q\d+")
    item_cols = [c for c in RAW.columns if pattern.match(str(c))]
    if not item_cols:
        raise RuntimeError("No questionnaire columns found (expected columns like Q1, Q2, ...).")

    # Keep only items
    X = RAW[item_cols].apply(pd.to_numeric, errors="coerce")

    # ---- Cleaning rules ----
    # 1) missingness <= 40%
    X = X.loc[:, X.isna().mean() <= 0.40]
    # 2) non-trivial variance
    X = X.loc[:, X.var(skipna=True) > 1e-6]

    # Store original index (rows) to keep alignment
    # If you want a specific ID column (e.g., country), set it here
    id_column_guess = None
    for guess in ["B_COUNTRY_ALPHA", "COUNTRY", "country", "Country"]:
        if guess in RAW.columns:
            id_column_guess = guess
            break

    # ---- Mean imputation & standardization ----
    X_imp = X.apply(lambda c: c.fillna(c.mean()))
    X_std = (X_imp - X_imp.mean()) / X_imp.std(ddof=0)
    X_std = X_std.fillna(0.0)  # safety
    n, p = X_std.shape

    print(f"[Info] n (countries) = {n}, p (items) = {p}")

    # ---- Adequacy measures on correlation matrix ----
    R = np.corrcoef(X_std, rowvar=False)
    kmo, chi2, df_b = kmo_bartlett(R, n, ridge=1e-6)
    print(f"[Adequacy] KMO ~ {kmo:.3f}, Bartlett chi2 ~ {chi2:.1f}, df={df_b}")

    # ---- Parallel Analysis ----
    obs_eigs, rand95, k_pa, k_kaiser = parallel_analysis(X_std, n_rep=100, seed=42, ridge=1e-6)
    print(f"[PA] Suggested factors (PA 95th): {k_pa}")
    print(f"[Kaiser] Eigenvalues > 1: {k_kaiser}")

    # Save Scree/PA plot
    scree_plot(obs_eigs, rand95, OUT / "scree_parallel.png")
    print("[Saved] scree_parallel.png")

    # ---- Model A: capacity-limited (cap at n-1) ----
    k_cap = max(1, min(k_pa, n - 1))
    print(f"[Model A] Fitting FactorAnalysis with k = {k_cap} (cap = n-1) ...")
    a_load, a_top, a_scores = fit_fa_with_rotation(
        X_std=X_std,
        k=k_cap,
        qmap=QMAP,
        id_col=id_column_guess,
        model_tag="rotated",           # consistent with earlier outputs
        out_dir=OUT
    )
    print(f"[Saved] {a_load.name}, {a_top.name}, {a_scores.name}")

    # ---- Model B: Parsimonious model (k=10) ----
    k_pars = min(10, n - 1)
    print(f"[Model B] Fitting FactorAnalysis with k = {k_pars} ...")
    b_load, b_top, b_scores = fit_fa_with_rotation(
        X_std=X_std,
        k=k_pars,
        qmap=QMAP,
        id_col=id_column_guess,
        model_tag="k10",
        out_dir=OUT
    )
    print(f"[Saved] {b_load.name}, {b_top.name}, {b_scores.name}")

    # ---- Summary printout ----
    print("\n=== SUMMARY ===")
    print(f"n={n}, p={p}")
    print(f"KMO ~ {kmo:.3f}, Bartlett chi2 ~ {chi2:.1f}, df={df_b}")
    print(f"PA suggests {k_pa} factors; Kaiser suggests {k_kaiser}.")
    print(f"Capacity-limited model used k={k_cap} (<= n-1).")
    print(f"Also saved a parsimonious k={k_pars} solution for interpretation.")


[Info] n (countries) = 66, p (items) = 315
[Adequacy] KMO ~ 0.990, Bartlett chi2 ~ -139256.3, df=49455
[PA] Suggested factors (PA 95th): 103
[Kaiser] Eigenvalues > 1: 45
[Saved] scree_parallel.png
[Model A] Fitting FactorAnalysis with k = 65 (cap = n-1) ...
[Saved] factor_loadings_rotated.csv, top_loadings_rotated.txt, factor_scores_rotated.csv
[Model B] Fitting FactorAnalysis with k = 10 ...
[Saved] factor_loadings_k10.csv, top_loadings_k10.txt, factor_scores_k10.csv

=== SUMMARY ===
n=66, p=315
KMO ~ 0.990, Bartlett chi2 ~ -139256.3, df=49455
PA suggests 103 factors; Kaiser suggests 45.
Capacity-limited model used k=65 (<= n-1).
Also saved a parsimonious k=10 solution for interpretation.


In [6]:

# ------------------------------------------------------------
# Factor analysis on WVS-style dataset with automatic factor naming
# Requirements: pandas, numpy, scikit-learn
# ------------------------------------------------------------
import json, re, warnings, math
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from sklearn.decomposition import FactorAnalysis

warnings.filterwarnings("ignore")

# -----------------------------
# 1) Load data and question map
# -----------------------------
CSV_PATH = Path("C:/Users/amaca253/Documents/datomatisation/datomatisation-dev/data/demo_data/wvs/data-wvs.csv")
MAP_PATH = Path("C:/Users/amaca253/Documents/datomatisation/datomatisation-dev/data/demo_data/wvs/map-wvs.json")

assert CSV_PATH.exists(), "data-wvs.csv not found"
assert MAP_PATH.exists(), "map-wvs.json not found"

raw = pd.read_csv(CSV_PATH, low_memory=False)
with open(MAP_PATH, "r", encoding="utf-8") as f:
    qmap = json.load(f)

# Keep only columns that have a question in the map (Q1..Q259)
item_cols = [c for c in raw.columns if c in qmap]
for c in item_cols:
    raw[c] = pd.to_numeric(raw[c], errors="coerce")
X = raw[item_cols].copy()

# -----------------------------------------
# 2) Basic cleaning and standardization
# -----------------------------------------
# Drop item columns with too many missing or quasi-constant
col_na_frac = X.isna().mean()
col_var = X.var(ddof=0)
keep_cols = [c for c in item_cols if col_na_frac[c] <= 0.40 and col_var[c] > 1e-6]
X = X[keep_cols]

# Drop rows with >30% missing among kept items; then impute medians
row_na_frac = X.isna().mean(axis=1)
X = X.loc[row_na_frac <= 0.30]
X = X.apply(lambda s: s.fillna(s.median()), axis=0)

# Z-score standardize
Xz = (X - X.mean()) / X.std(ddof=0)
Xz = Xz.replace([np.inf, -np.inf], np.nan).fillna(0)

n, p = Xz.shape
print(f"n_observations={n}, n_items={p}")

# -----------------------------------------
# 3) Determine number of factors (k)
#    - Parallel Analysis (Horn)
#    - Velicer's MAP
#    - Safety cap for p >> n
# -----------------------------------------
R = np.corrcoef(Xz.values, rowvar=False)
vals, vecs = np.linalg.eigh(R)
idx = np.argsort(vals)[::-1]
vals = vals[idx]; vecs = vecs[:, idx]

# Velicer's MAP
def map_test(R, vals, vecs, m_max=30):
    p = R.shape[0]
    m_max = min(m_max, p-1)
    scores = []
    for m in range(m_max+1):
        if m == 0:
            resid = R.copy()
        else:
            L = vecs[:, :m] @ np.diag(vals[:m]) @ vecs[:, :m].T
            resid = R - L
        np.fill_diagonal(resid, 0)
        scores.append(np.mean(resid**2))
    return int(np.argmin(scores)), scores

k_map, map_scores = map_test(R, vals, vecs, m_max=30)

# Parallel Analysis (95th percentile of random eigenvalues)
np.random.seed(42)
reps = 50
rand_eigs = np.zeros((reps, p))
for i in range(reps):
    Z = np.random.normal(size=(n, p))
    Z = (Z - Z.mean(axis=0)) / Z.std(axis=0)
    Rz = np.corrcoef(Z, rowvar=False)
    w, _ = np.linalg.eigh(Rz)
    rand_eigs[i, :] = np.sort(w)[::-1]
rand_cut = np.percentile(rand_eigs, 95, axis=0)
obs_eigs = np.sort(np.linalg.eigvalsh(R))[::-1]
k_par = int(np.sum(obs_eigs > rand_cut))

# Practical cap to avoid overfitting with p >> n
k_cap = min(12, max(1, n // 4), p - 1)
k_final = max(1, min(k_map, k_par, k_cap))

print(f"k_map={k_map}, k_parallel={k_par}, k_cap={k_cap} -> k_final={k_final}")

# -----------------------------------------
# 4) Factor analysis + Varimax rotation
# -----------------------------------------
def varimax(Phi, gamma=1.0, q=50, tol=1e-6):
    p, k = Phi.shape
    R = np.eye(k)
    d = 0
    for _ in range(q):
        d_old = d
        Lambda = Phi @ R
        U, S, Vt = np.linalg.svd(
            Phi.T @ (Lambda**3 - (gamma/p)*Lambda @ np.diag(np.diag(Lambda.T @ Lambda)))
        )
        R = U @ Vt
        d = np.sum(S)
        if d_old != 0 and d/d_old < 1 + tol:
            break
    return Phi @ R, R

fa = FactorAnalysis(n_components=k_final, random_state=42)
F = fa.fit_transform(Xz.values)
loadings = fa.components_.T  # shape p x k

rot_load, rot_mat = varimax(loadings)

# -----------------------------------------
# 5) Automatic factor naming
# -----------------------------------------
keyword_groups = {
    'Religion & Faith': ['religion','religious','god','pray','church','faith','atheist'],
    'Trust & Institutions': ['trust','confidence','police','courts','government','parliament','banks','companies',
                             'united nations','european union','imf','nato','who','wto','elections','parties',
                             'universities','press','television','tv','labor unions','army','civil service'],
    'Family & Parenting': ['family','parents','children','marriage','housewife','home','mother','father'],
    'Work Ethic & Duty': ['work','jobs','unemployment','duty','hard work','competition','business','executives'],
    'Immigration & Diversity': ['immigration','immigrants','foreign','refugees','diversity','nationality','language','race'],
    'Security & Crime': ['security','crime','terrorism','war','violence','harassment','unsafe','weapon','police','military'],
    'Science & Technology': ['science','technology'],
    'Democracy & Politics': ['democracy','vote','politics','freedom','equality','left','right','leader','experts','army rule','religious law','political system'],
    'Tolerance & Social Norms': ['homosexual','neighbors','tolerance','respect','different religion','different race','unmarried couples'],
    'Economy & Redistribution': ['income','economy','prices','tax','ownership','market','growth','environment','equal','incentives','subsidize','aid'],
    'Wellbeing & Satisfaction': ['happy','happiness','health','satisfied','life','financial','standard of living','free choice','control'],
    'Corruption & Bribery': ['corruption','bribe','gift','favor','risk','accountable'],
    'Civic Participation & Media': ['petition','boycotts','demonstrations','strikes','donated','contacted','encouraged','vote','social media',
                                    'newspapers','radio','tv news','internet','email','mobile'],
    'Moral Permissiveness': ['justified','abortion','divorce','suicide','euthanasia','sex','prostitution','homosexuality','bribe','cheating','stealing'],
    'Identity & Belonging': ['proud','nationality','village','town','city','country','continent','world','close'],
}

qtext = {k: qmap[k].lower() for k in Xz.columns if k in qmap}
thr = 0.40
abs_load = np.abs(rot_load)

rows = []
for j in range(k_final):
    # salient items
    idxs = np.where(abs_load[:, j] >= thr)[0]
    items = [Xz.columns[i] for i in idxs]

    # naming by keyword counts over top items' question text
    bag = " ".join([qtext.get(c, c) for c in items])
    scores = {name: sum(bag.count(kw) for kw in kws) for name, kws in keyword_groups.items()}
    best = [name for name, sc in sorted(scores.items(), key=lambda x: -x[1])[:2] if sc > 0]
    proposed_name = " + ".join(best) if best else f"Factor {j+1}"

    # capture top loadings (absolute)
    series_j = pd.Series(rot_load[:, j], index=Xz.columns)
    top = series_j.abs().sort_values(ascending=False).head(12)
    for q in top.index:
        rows.append({
            "factor": j+1,
            "factor_name": proposed_name,
            "item": q,
            "loading": float(series_j[q]),
            "question": qmap.get(q, "")
        })

# Save a tidy table of the rotated loadings
out_df = pd.DataFrame(rows)
out_df.to_csv("rotated_loadings.csv", index=False)

# Pretty print a compact summary
print("\n=== Compact factor summary (top items) ===")
for j in range(1, k_final+1):
    sub = out_df[out_df["factor"] == j].copy()
    name = sub["factor_name"].iloc[0]
    # Keep only the top 6 absolute loadings for display
    sub["abs"] = sub["loading"].abs()
    sub = sub.sort_values("abs", ascending=False).head(6)
    print(f"\nFactor {j}: {name}")
    for _, r in sub.iterrows():
        print(f"  {r['item']:>4s}  {r['loading']:+.3f}  {r['question']}")
print("\nWrote rotated loadings to: rotated_loadings.csv")


n_observations=66, n_items=259
k_map=30, k_parallel=111, k_cap=12 -> k_final=12

=== Compact factor summary (top items) ===

Factor 1: Trust & Institutions + Identity & Belonging
  Q184  +0.925  Please tell me for the following action whether you think it can always be justified, never be justified, or something in between: Abortion.
  Q186  +0.923  Please tell me for the following action whether you think it can always be justified, never be justified, or something in between: Sex before marriage.
   Q38  +0.918  How would you feel about the following statements? Do you agree or disagree? Adult children have the duty to provide long-term care for their parents.
  Q185  +0.913  Please tell me for the following action whether you think it can always be justified, never be justified, or something in between: Divorce.
  Q182  +0.913  Please tell me for the following action whether you think it can always be justified, never be justified, or something in between: Homosexuality.
  Q188  +0.

In [8]:

# ------------------------------------------------------------
# Cluster analysis on rotated factor scores with auto-naming
# ------------------------------------------------------------
import json, re, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.decomposition import FactorAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

warnings.filterwarnings('ignore')

# -----------------
# Load data & map
# -----------------
CSV = Path("/data/demo_data/wvs/data-wvs.csv")
MAP = Path("/data/demo_data/wvs/map-wvs.json")

raw = pd.read_csv(CSV, low_memory=False)
with open(MAP, "r", encoding="utf-8") as f:
    qmap = json.load(f)

# keep only items in the map (Q1..Q259)
item_cols = [c for c in raw.columns if c in qmap]
for c in item_cols:
    raw[c] = pd.to_numeric(raw[c], errors="coerce")
X = raw[item_cols].copy()

# -------------
# Cleaning
# -------------
col_na_frac = X.isna().mean()
col_var = X.var(ddof=0)
keep_cols = [c for c in item_cols if col_na_frac[c] <= 0.40 and col_var[c] > 1e-6]
X = X[keep_cols]

row_na_frac = X.isna().mean(axis=1)
X = X.loc[row_na_frac <= 0.30]
X = X.apply(lambda s: s.fillna(s.median()), axis=0)

# standardize
Xz = (X - X.mean()) / X.std(ddof=0)
Xz = Xz.replace([np.inf, -np.inf], np.nan).fillna(0)

n, p = Xz.shape

# ---------------------------------------------------
# Factor analysis with Varimax rotation (orthogonal)
# ---------------------------------------------------
# cap factors for stability when p >> n
k = min(12, max(1, n // 4), p - 1)

fa = FactorAnalysis(n_components=k, random_state=42)
F = fa.fit_transform(Xz.values)     # unrotated scores (n x k)
L = fa.components_.T                # loadings (p x k)

def varimax(Phi, gamma=1.0, q=50, tol=1e-6):
    """Orthogonal Varimax rotation."""
    p, k = Phi.shape
    R = np.eye(k)
    d = 0
    for _ in range(q):
        d_old = d
        Lambda = Phi @ R
        U, S, Vt = np.linalg.svd(
            Phi.T @ (Lambda**3 - (gamma/p) * Lambda @ np.diag(np.diag(Lambda.T @ Lambda)))
        )
        R = U @ Vt
        d = np.sum(S)
        if d_old != 0 and d/d_old < 1 + tol:
            break
    return Phi @ R, R

L_rot, R = varimax(L)
F_rot = F @ R  # rotate scores to match rotated loadings

# -----------------------------------------
# Factor names via question-text keywords
# -----------------------------------------
keyword_groups = {
    'Religion & Faith': ['religion','religious','god','pray','church','faith','atheist'],
    'Trust & Institutions': ['trust','confidence','police','courts','government','parliament','banks','companies',
                             'united nations','european union','imf','nato','who','wto','elections','parties',
                             'universities','press','television','tv','labor unions','army','civil service'],
    'Family & Parenting': ['family','parents','children','marriage','housewife','home','mother','father'],
    'Work Ethic & Duty': ['work','jobs','unemployment','duty','hard work','competition','business','executives'],
    'Immigration & Diversity': ['immigration','immigrants','foreign','refugees','diversity','nationality','language','race'],
    'Security & Crime': ['security','crime','terrorism','war','violence','harassment','unsafe','weapon','police','military'],
    'Science & Technology': ['science','technology'],
    'Democracy & Politics': ['democracy','vote','politics','freedom','equality','left','right','leader','experts',
                             'army rule','religious law','political system'],
    'Tolerance & Social Norms': ['homosexual','neighbors','tolerance','respect','different religion','different race','unmarried couples'],
    'Economy & Redistribution': ['income','economy','prices','tax','ownership','market','growth','environment','equal',
                                 'incentives','subsidize','aid'],
    'Wellbeing & Satisfaction': ['happy','happiness','health','satisfied','life','financial','standard of living','free choice','control'],
    'Corruption & Bribery': ['corruption','bribe','gift','favor','risk','accountable'],
    'Civic Participation & Media': ['petition','boycotts','demonstrations','strikes','donated','contacted','encouraged','vote',
                                    'social media','newspapers','radio','tv news','internet','email','mobile'],
    'Moral Permissiveness': ['justified','abortion','divorce','suicide','euthanasia','sex','prostitution','homosexuality','bribe','cheating','stealing'],
    'Identity & Belonging': ['proud','nationality','village','town','city','country','continent','world','close'],
}

qtext = {q: qmap[q].lower() for q in Xz.columns if q in qmap}
absL = np.abs(L_rot); thr = 0.40
factor_names = []
for j in range(k):
    idxs = np.where(absL[:, j] >= thr)[0]
    cols = [Xz.columns[i] for i in idxs]
    bag = " ".join([qtext.get(c, c) for c in cols])
    scores = {nm: sum(bag.count(kw) for kw in kws) for nm, kws in keyword_groups.items()}
    top2 = [nm for nm, sc in sorted(scores.items(), key=lambda x: -x[1])[:2] if sc > 0]
    factor_names.append(" + ".join(top2) if top2 else f"Factor {j+1}")

# save factor scores
F_df = pd.DataFrame(F_rot, columns=[f"F{j+1}_{factor_names[j]}" for j in range(k)])
id_col = None
for cand in ["B_COUNTRY_ALPHA", "Country", "country", "COUNTRY", "Name"]:
    if cand in raw.columns:
        id_col = cand; break
if id_col:
    F_df.insert(0, id_col, raw.loc[F_df.index, id_col].values)
F_df.to_csv("factor_scores_rotated.csv", index=False)

# -----------------------------------------
# Cluster search: 5 metrics + voting
# -----------------------------------------
Z = StandardScaler().fit_transform(F_rot)
k_min, k_max = 2, min(10, n-1)
results = []

def gap_statistic(X, n_refs=20, k_range=range(2, 11), random_state=42):
    rng = np.random.RandomState(random_state)
    mins, maxs = X.min(axis=0), X.max(axis=0)
    gaps = {}
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=25).fit(X)
        disp = np.sum((X - km.cluster_centers_[km.labels_])**2)
        ref_log = []
        for _ in range(n_refs):
            X_ref = rng.uniform(mins, maxs, size=X.shape)
            km_ref = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_ref)
            ref_disp = np.sum((X_ref - km_ref.cluster_centers_[km_ref.labels_])**2)
            ref_log.append(np.log(ref_disp))
        gaps[k] = np.mean(ref_log) - np.log(disp)
    return gaps

gaps = gap_statistic(Z, n_refs=30, k_range=range(k_min, k_max+1))

for k_try in range(k_min, k_max+1):
    km = KMeans(n_clusters=k_try, random_state=42, n_init=50)
    labels = km.fit_predict(Z)
    sil = silhouette_score(Z, labels)
    ch  = calinski_harabasz_score(Z, labels)
    db  = davies_bouldin_score(Z, labels)
    gmm = GaussianMixture(n_components=k_try, covariance_type="full", random_state=42).fit(Z)
    bic = gmm.bic(Z)
    results.append({"k": k_try, "sil": sil, "ch": ch, "db": db, "gap": gaps[k_try], "bic": bic})

res_df = pd.DataFrame(results)
res_df["sil_rank"] = res_df["sil"].rank(ascending=False)
res_df["ch_rank"]  = res_df["ch"].rank(ascending=False)
res_df["gap_rank"] = res_df["gap"].rank(ascending=False)
res_df["db_rank"]  = res_df["db"].rank(ascending=True)
res_df["bic_rank"] = res_df["bic"].rank(ascending=True)
res_df["total_rank"] = res_df[["sil_rank","ch_rank","gap_rank","db_rank","bic_rank"]].mean(axis=1)
best_k = int(res_df.sort_values(["total_rank","sil_rank"]).iloc[0]["k"])

# final model
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=100)
labels = kmeans.fit_predict(Z)
centroids = pd.DataFrame(kmeans.cluster_centers_, columns=[f"F{j+1}" for j in range(k)])
centroids_named = centroids.copy()
centroids_named.columns = [f"F{j+1}_{factor_names[j]}" for j in range(k)]

# -----------------------------------------
# Auto-name clusters + 3-sentence description
# -----------------------------------------
pos_top_n, neg_top_n = 2, 1
cluster_info = []
for c in range(best_k):
    row = centroids_named.iloc[c]
    pos = row.sort_values(ascending=False).head(pos_top_n)
    neg = row.sort_values(ascending=True).head(neg_top_n)
    pos_names = [re.sub(r"^F\\d+_", "", nm) for nm in pos.index]
    neg_names = [re.sub(r"^F\\d+_", "", nm) for nm in neg.index]

    name = "High " + " & ".join(pos_names)
    if neg_names: name += ", Low " + " & ".join(neg_names)

    # overview
    n_members = int(np.sum(labels == c))
    overview = (f"This cluster groups {n_members} units with above-average scores on "
                f"{', '.join(pos_names)} and below-average levels on {', '.join(neg_names)}.")
    # strengths / weaknesses thresholds
    strengths_vars = [re.sub(r"^F\\d+_", "", nm) for nm, val in row.items() if val >= 0.40]
    weaknesses_vars = [re.sub(r"^F\\d+_", "", nm) for nm, val in row.items() if val <= -0.40]

    strengths = ("Strengths include relatively high levels of " + ", ".join(strengths_vars) + ".") \
                if strengths_vars else "Strengths are not pronounced; factor scores are near the sample average."
    weaknesses = ("Weaknesses center on comparatively low levels of " + ", ".join(weaknesses_vars) + ".") \
                 if weaknesses_vars else "Few clear weaknesses emerge; low scores are modest in magnitude."

    cluster_info.append({
        "cluster": int(c),
        "name": name,
        "overview": overview,
        "strengths": strengths,
        "weaknesses": weaknesses
    })

# -----------------------------------------
# Save outputs
# -----------------------------------------
assign = pd.DataFrame({"cluster": labels})
if id_col: assign[id_col] = raw.loc[assign.index, id_col].values
assign = assign[[col for col in ([id_col, "cluster"] if id_col else ["cluster"])]]
assign.to_csv("cluster_assignments.csv", index=False)

centroids_named.to_csv("cluster_centroids_zscores.csv", index_label="cluster")
res_df.to_csv("cluster_model_selection.csv", index=False)
with open("cluster_info.json", "w", encoding="utf-8") as f:
    json.dump({"best_k": best_k, "factor_names": factor_names, "clusters": cluster_info},
              f, ensure_ascii=False, indent=2)

print(f"Best k = {best_k}")
print("Files written: factor_scores_rotated.csv, cluster_model_selection.csv, "
      "cluster_centroids_zscores.csv, cluster_assignments.csv, cluster_info.json")


Best k = 10
Files written: factor_scores_rotated.csv, cluster_model_selection.csv, cluster_centroids_zscores.csv, cluster_assignments.csv, cluster_info.json


In [9]:
cluster_info

[{'cluster': 0,
  'name': 'High F10_Trust & Institutions + Civic Participation & Media & F4_Economy & Redistribution + Civic Participation & Media, Low F8_Democracy & Politics + Civic Participation & Media',
  'overview': 'This cluster groups 2 units with above-average scores on F10_Trust & Institutions + Civic Participation & Media, F4_Economy & Redistribution + Civic Participation & Media and below-average levels on F8_Democracy & Politics + Civic Participation & Media.',
  'strengths': 'Strengths include relatively high levels of F2_Trust & Institutions + Identity & Belonging, F4_Economy & Redistribution + Civic Participation & Media, F10_Trust & Institutions + Civic Participation & Media.',
  'weaknesses': 'Weaknesses center on comparatively low levels of F1_Trust & Institutions + Identity & Belonging, F3_Moral Permissiveness + Democracy & Politics, F5_Trust & Institutions + Security & Crime, F6_Religion & Faith + Wellbeing & Satisfaction, F7_Wellbeing & Satisfaction + Economy & Re

In [10]:

# ------------------------------------------------------------
# Concise 3-sentence country summary from factor & cluster model
# ------------------------------------------------------------
import json, re, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.decomposition import FactorAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

warnings.filterwarnings('ignore')

# -----------------
# User settings
# -----------------
COUNTRY = "Argentina"   # <- change to any country present in data-wvs.csv



# -----------------
# Load data & map
# -----------------
raw = pd.read_csv(CSV, low_memory=False)
with open(MAP, "r", encoding="utf-8") as f:
    qmap = json.load(f)

# keep only items listed in map (Q1..Q259)
item_cols = [c for c in raw.columns if c in qmap]
for c in item_cols:
    raw[c] = pd.to_numeric(raw[c], errors="coerce")
X = raw[item_cols].copy()

# -----------------
# Cleaning
# -----------------
col_na = X.isna().mean()
col_var = X.var(ddof=0)
keep_cols = [c for c in item_cols if col_na[c] <= 0.40 and col_var[c] > 1e-6]
X = X[keep_cols]

row_na = X.isna().mean(axis=1)
X = X.loc[row_na <= 0.30]
X = X.apply(lambda s: s.fillna(s.median()), axis=0)

# standardize items
Xz = (X - X.mean()) / X.std(ddof=0)
Xz = Xz.replace([np.inf, -np.inf], np.nan).fillna(0)

n, p = Xz.shape

# -----------------
# Factor analysis (MLE) + Varimax rotation
# -----------------
k = min(12, max(1, n // 4), p - 1)     # practical cap for p >> n
fa = FactorAnalysis(n_components=k, random_state=42)
F = fa.fit_transform(Xz.values)        # scores (unrotated)
L = fa.components_.T                   # loadings (p x k)

def varimax(Phi, gamma=1.0, q=50, tol=1e-6):
    p, k = Phi.shape
    R = np.eye(k)
    d = 0
    for _ in range(q):
        d_old = d
        Lambda = Phi @ R
        U, S, Vt = np.linalg.svd(
            Phi.T @ (Lambda**3 - (gamma/p) * Lambda @ np.diag(np.diag(Lambda.T @ Lambda)))
        )
        R = U @ Vt
        d = np.sum(S)
        if d_old != 0 and d/d_old < 1 + tol:
            break
    return Phi @ R, R

L_rot, R = varimax(L)
F_rot = F @ R  # rotated factor scores

# -----------------
# Auto naming of factors using question text keywords
# -----------------
keyword_groups = {
    'Religion & Faith': ['religion','religious','god','pray','church','faith','atheist'],
    'Trust & Institutions': ['trust','confidence','police','courts','government','parliament','banks','companies',
                             'united nations','european union','imf','nato','who','wto','elections','parties',
                             'universities','press','television','tv','labor unions','army','civil service'],
    'Family & Parenting': ['family','parents','children','marriage','housewife','home','mother','father'],
    'Work Ethic & Duty': ['work','jobs','unemployment','duty','hard work','competition','business','executives'],
    'Immigration & Diversity': ['immigration','immigrants','foreign','refugees','diversity','nationality','language','race'],
    'Security & Crime': ['security','crime','terrorism','war','violence','harassment','unsafe','weapon','police','military'],
    'Science & Technology': ['science','technology'],
    'Democracy & Politics': ['democracy','vote','politics','freedom','equality','left','right','leader','experts',
                             'army rule','religious law','political system'],
    'Tolerance & Social Norms': ['homosexual','neighbors','tolerance','respect','different religion','different race','unmarried couples'],
    'Economy & Redistribution': ['income','economy','prices','tax','ownership','market','growth','environment','equal',
                                 'incentives','subsidize','aid'],
    'Wellbeing & Satisfaction': ['happy','happiness','health','satisfied','life','financial','standard of living','free choice','control'],
    'Corruption & Bribery': ['corruption','bribe','gift','favor','risk','accountable'],
    'Civic Participation & Media': ['petition','boycotts','demonstrations','strikes','donated','contacted','encouraged','vote',
                                    'social media','newspapers','radio','tv news','internet','email','mobile'],
    'Moral Permissiveness': ['justified','abortion','divorce','suicide','euthanasia','sex','prostitution','homosexuality','bribe','cheating','stealing'],
    'Identity & Belonging': ['proud','nationality','village','town','city','country','continent','world','close'],
}
qtext = {q: qmap[q].lower() for q in Xz.columns if q in qmap}
absL = np.abs(L_rot); thr = 0.40
factor_names = []
for j in range(k):
    idxs = np.where(absL[:, j] >= thr)[0]
    cols = [Xz.columns[i] for i in idxs]
    bag = " ".join([qtext.get(c, c) for c in cols])
    scores = {nm: sum(bag.count(kw) for kw in kws) for nm, kws in keyword_groups.items()}
    top2 = [nm for nm, sc in sorted(scores.items(), key=lambda x: -x[1])[:2] if sc > 0]
    factor_names.append(" + ".join(top2) if top2 else f"Factor {j+1}")

F_df = pd.DataFrame(F_rot, columns=[f"F{j+1}_{factor_names[j]}" for j in range(k)])

# attach an id column if present
id_col = None
for cand in ["B_COUNTRY_ALPHA","Country","country","COUNTRY","Name"]:
    if cand in raw.columns:
        id_col = cand; break
if id_col:
    F_df.insert(0, id_col, raw.loc[F_df.index, id_col].values)

# -----------------
# Find optimal number of clusters on factor scores
# -----------------
Z = StandardScaler().fit_transform(F_rot)
k_min, k_max = 2, min(10, n-1)
results = []

def gap_statistic(X, n_refs=20, k_range=range(2,11), random_state=42):
    rng = np.random.RandomState(random_state)
    mins, maxs = X.min(axis=0), X.max(axis=0)
    gaps = {}
    for kk in k_range:
        km = KMeans(n_clusters=kk, random_state=42, n_init=50).fit(X)
        disp = np.sum((X - km.cluster_centers_[km.labels_])**2)
        ref_log = []
        for _ in range(n_refs):
            X_ref = rng.uniform(mins, maxs, size=X.shape)
            km_ref = KMeans(n_clusters=kk, random_state=42, n_init=10).fit(X_ref)
            ref_log.append(np.log(np.sum((X_ref - km_ref.cluster_centers_[km_ref.labels_])**2)))
        gaps[kk] = np.mean(ref_log) - np.log(disp)
    return gaps

gaps = gap_statistic(Z, n_refs=20, k_range=range(k_min, k_max+1))
for kk in range(k_min, k_max+1):
    km = KMeans(n_clusters=kk, random_state=42, n_init=50)
    labels = km.fit_predict(Z)
    sil = silhouette_score(Z, labels)
    ch  = calinski_harabasz_score(Z, labels)
    db  = davies_bouldin_score(Z, labels)
    bic = GaussianMixture(n_components=kk, covariance_type="full", random_state=42).fit(Z).bic(Z)
    results.append({"k": kk, "sil": sil, "ch": ch, "db": db, "gap": gaps[kk], "bic": bic})

res_df = pd.DataFrame(results)
res_df["sil_rank"] = res_df["sil"].rank(ascending=False)
res_df["ch_rank"]  = res_df["ch"].rank(ascending=False)
res_df["gap_rank"] = res_df["gap"].rank(ascending=False)
res_df["db_rank"]  = res_df["db"].rank(ascending=True)
res_df["bic_rank"] = res_df["bic"].rank(ascending=True)
res_df["total_rank"] = res_df[["sil_rank","ch_rank","gap_rank","db_rank","bic_rank"]].mean(axis=1)
best_k = int(res_df.sort_values(["total_rank","sil_rank"]).iloc[0]["k"])

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=100)
labels = kmeans.fit_predict(Z)

# -----------------
# Build the 3-sentence summary for COUNTRY
# -----------------
if id_col:
    sel = F_df[id_col].astype(str).str.lower().eq(COUNTRY.lower())
    if not sel.any():
        sel = F_df[id_col].astype(str).str.upper().eq(COUNTRY.upper()[:3])  # fallback arg code
else:
    raise RuntimeError("No country column found in data-wvs.csv")

assert sel.any(), f"{COUNTRY} not found."
idx = sel.idxmax()
row_scores = F_df.loc[idx].drop(labels=[id_col])

# strengths (>= +0.35 z), weaknesses (<= -0.35 z), averages (|z|<0.25)
clean = lambda s: re.sub(r"^F\\d+_", "", s)
s_sorted = row_scores.sort_values(ascending=False)
w_sorted = row_scores.sort_values(ascending=True)
pos_salient = [clean(k) for k, v in s_sorted.items() if v >= 0.35][:3]
neg_salient = [clean(k) for k, v in w_sorted.items() if v <= -0.35][:2]
avg_list   = [clean(k) for k, v in row_scores.items() if abs(v) < 0.25][:2]

# simple cluster label from centroid structure
cent = pd.DataFrame(kmeans.cluster_centers_, columns=F_df.columns[1:])
c_id = int(labels[idx])
cent_row = cent.iloc[c_id]
pos_top = [clean(nm) for nm in cent_row.sort_values(ascending=False).head(2).index]
neg_top = [clean(nm) for nm in cent_row.sort_values(ascending=True).head(1).index]
cluster_name = "High " + " & ".join(pos_top)
if neg_top:
    cluster_name += ", Low " + " & ".join(neg_top)

# sentences
s1 = "Argentina stands out for relatively high scores on " + ", ".join(pos_salient or [clean(s_sorted.index[0])]) + "."
if neg_salient and avg_list:
    s2 = ("It is average on " + ", ".join(avg_list) +
          ", while showing comparatively lower scores on " + ", ".join(neg_salient) + ".")
elif neg_salient:
    s2 = "It shows comparatively lower scores on " + ", ".join(neg_salient) + "."
else:
    s2 = "Most remaining dimensions are around the sample average."
s3 = f"Overall, Argentina fits the profile '{cluster_name}', combining its strengths with areas for improvement."

print(s1)
print(s2)
print(s3)


Argentina stands out for relatively high scores on F8_Democracy & Politics + Civic Participation & Media, F5_Trust & Institutions + Security & Crime, F1_Trust & Institutions + Identity & Belonging.
It is average on F3_Moral Permissiveness + Democracy & Politics, F9_Tolerance & Social Norms + Immigration & Diversity, while showing comparatively lower scores on F10_Trust & Institutions + Civic Participation & Media, F11_Identity & Belonging + Trust & Institutions.
Overall, Argentina fits the profile 'High F5_Trust & Institutions + Security & Crime & F11_Identity & Belonging + Trust & Institutions, Low F7_Wellbeing & Satisfaction + Economy & Redistribution', combining its strengths with areas for improvement.


In [ ]:

import re
import pandas as pd

# Example: after you build factor score DataFrame F_df
# Columns look like: F1_Trust & Institutions, F2_Religion & Faith, ...

def clean_factor_names(cols):
    return [re.sub(r'^F\d+_', '', c) for c in cols]

F_df.columns = ([F_df.columns[0]] + clean_factor_names(F_df.columns[1:])) if F_df.columns[0] not in ['cluster'] else clean_factor_names(F_df.columns)

F_df = pd.DataFrame(F_rot, columns=[factor_names[j] for j in range(k)])  # no F{j+1}_ prefix





def strip_prefix(df):
    new_cols = []
    for c in df.columns:
        # do not touch id/label columns you recognize
        if c in ['B_COUNTRY_ALPHA','Country','country','COUNTRY','Name','cluster']:
            new_cols.append(c)
        else:
            new_cols.append(re.sub(r'^F\\d+_', '', c))
    df.columns = new_cols
    return df

# Factor scores
fs = pd.read_csv('factor_scores_rotated.csv')
fs = strip_prefix(fs)
fs.to_csv('factor_scores_rotated_clean.csv', index=False)

# Cluster centroids (z-scores)
cent = pd.read_csv('cluster_centroids_zscores.csv', index_col=0)
cent = strip_prefix(cent)
cent.to_csv('cluster_centroids_zscores_clean.csv')




def clean_name(s): 
    return re.sub(r'^F\\d+_', '', s)

# Example: when creating the label
pos_top = [ clean_name(nm) for nm in rowc.sort_values(ascending=False).head(2).index ]
neg_top = [ clean_name(nm) for nm in rowc.sort_values(ascending=True).head(1).index ]
cluster_name = "High " + " & ".join(pos_top)
if neg_top:
    cluster_name += ", Low " + " & ".join(neg_top)

